In [1]:
exec(open(__import__('pathlib').Path(__vsc_ipynb_file__).parent.parent / 'src' / 'display_html.py').read())

## L_ECin / L_ECout — Entorhinal Cortex I/O

**Role**: Cortical gateway in and out of the hippocampal circuit (Schapiro 2017 §2.a.ii–iv).  
`L_ECin` drives all hippocampal layers with the external stimulus; `L_ECout` reconstructs  
the predicted next item from CA1 activity. In the plus phase, ECout is clamped to the  
target — its activity back-projects to CA1 via `W_ECout` as the CHL teaching signal.

**Key parameters (Schapiro 2017 §2.a.ii)**:
- `k = 2` (absolute): exactly 2 units active in both ECin and ECout at any time
- `n_items = 15` (community task): 2/15 ≈ 13% active
- `lr = 0.05` (MSP rate): W_ECout big-loop and W_CA1_ECout updated slowly
- ECin moving window (§2.c): `clamp = one_hot(curr)*1.0 + one_hot(prev)*0.9`
- Big loop: `W_ECout` shape (n_ECout, n_units) lives in L_ECin; scale = 2.0 (SI Table 2)
- `use_euler=False` default for L_ECin: snaps immediately (Euler only needed with big loop)

**Why k=2 absolute (not fractional)?**  
ECin and ECout represent single items. The moving window encodes exactly two items  
(curr at 1.0, prev at 0.9); k=2 selects them. With a pure one-hot (one item only),  
the threshold lands at 0.0 and only that unit stays active — kWTA degrades gracefully.

**Understanding check**: In the minus phase, ECout settles freely from CA1 input.  
In the plus phase, ECout is clamped to the target. Why does this matter for CHL?  
→ ΔW = lr × (ActP ⊗ ActP − ActM ⊗ ActM). ECout minus = network prediction (ActM);  
ECout plus = correct answer (ActP). The difference drives learning in W_ECin, W_CA3, W_ECout.

In [2]:
# path & directories
import sys
from pathlib import Path

DIR_SRC = str((Path(__vsc_ipynb_file__).parent.parent / 'src').resolve())
DIR_VIZ = (Path(__vsc_ipynb_file__).parent.parent / 'visualizations').resolve()
sys.path.insert(0, DIR_SRC)

# imports
import torch
from layer import L_ECin, L_ECout
from util import NET_SCALE

# Schapiro (2017) §2.a.ii community task parameters
N_ITEMS = 15   # Schapiro (2017) Fig. 1
N_CA1   = 50   # test scale

torch.manual_seed(42)

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


## L_ECin — Basic Forward and kWTA

In [3]:
# GOAL: Confirm L_ECin instantiates with k=2 absolute kWTA and no big loop by default
# Instantiate without big loop (n_ECout=None; default)
ecin = L_ECin(n_units=N_ITEMS, k=2, use_euler=False)

print(f"n_units : {ecin.n_units}")
print(f"k       : {ecin.k}  (absolute; Schapiro 2017 §2.a.ii)")
print(f"W_ECout : {ecin.W_ECout}  (None — big loop disabled)")
print(f"k/n     : {ecin.k}/{ecin.n_units} = {ecin.k/ecin.n_units:.3f}  (~13% active)")

n_units : 15
k       : 2  (absolute; Schapiro 2017 §2.a.ii)
W_ECout : None  (None — big loop disabled)
k/n     : 2/15 = 0.133  (~13% active)


In [4]:
# GOAL: Confirm k=2 kWTA selects exactly 2 units from moving window (curr=1.0, prev=0.9)
# k=2 absolute kWTA — test with one-hot vs moving window
# One-hot: only curr item → threshold lands at 0.0; only the positive unit survives
clamp_onehot = torch.zeros(N_ITEMS)
clamp_onehot[0] = 1.0

act_onehot = ecin(clamp_onehot)
n_onehot = (act_onehot > 0).sum().item()
print(f"One-hot (curr=1.0 only):       active = {n_onehot}  (1 — threshold at 0.0, zeros survive as 0)")

# Moving window: curr=1.0, prev=0.9 → threshold = 0.9 → exactly 2 units > 0
# Schapiro (2017) §2.c: ECin receives curr item at 1.0 and prev item at 0.9
clamp_window = torch.zeros(N_ITEMS)
clamp_window[0] = 1.0   # curr item
clamp_window[1] = 0.9   # prev item

act_window = ecin(clamp_window)
n_window = (act_window > 0).sum().item()
print(f"Moving window (curr=1.0, prev=0.9): active = {n_window}  (expect 2)")
print(f"  Unit 0 (curr): {act_window[0].item():.3f}")
print(f"  Unit 1 (prev): {act_window[1].item():.3f}")
print("Moving window is required for k=2 to produce exactly 2 active units.")

One-hot (curr=1.0 only):       active = 1  (1 — threshold at 0.0, zeros survive as 0)
Moving window (curr=1.0, prev=0.9): active = 2  (expect 2)
  Unit 0 (curr): 0.998
  Unit 1 (prev): 0.997
Moving window is required for k=2 to produce exactly 2 active units.


## L_ECin — Big Loop (ECout back-projection)

In [5]:
# GOAL: Confirm W_ECout shape, init range, and that ECout back-projection modifies net input
# Big loop: instantiate with n_ECout → W_ECout created
# W_ECout shape (n_ECout, n_units); init uniform(0.49, 0.51) — SI Table 2 'big_loop'
ecin_big = L_ECin(n_units=N_ITEMS, n_ECout=N_ITEMS, k=2, use_euler=True)

print(f"W_ECout shape : {ecin_big.W_ECout.shape}  (n_ECout={N_ITEMS}, n_units={N_ITEMS})")
print(f"W_ECout range : [{ecin_big.W_ECout.min():.4f}, {ecin_big.W_ECout.max():.4f}]")
print(f"  (expect ~uniform(0.49, 0.51) — Schapiro 2017 SI Table 2 big_loop init)")
print(f"NET_SCALE ecout_ecin = {NET_SCALE['ecout_ecin']}  (SI Table 2 abs=2)")
print()

# With a_ECout: net = clamp + 2.0 * (a_ECout @ W_ECout)
a_ecout_zero = torch.zeros(N_ITEMS)
a_ecout_item = torch.zeros(N_ITEMS)
a_ecout_item[3] = 1.0   # ECout unit 3 active

ecin_big.reset()
act_no_back   = ecin_big(clamp_window, a_ecout_zero)
ecin_big.reset()
act_with_back = ecin_big(clamp_window, a_ecout_item)

same = torch.allclose(act_no_back, act_with_back)
print(f"Active (no back-proj) : {(act_no_back   > 0).nonzero().squeeze().tolist()}")
print(f"Active (with ECout[3]): {(act_with_back > 0).nonzero().squeeze().tolist()}")
print(f"Outputs differ: {not same}  (expect True — back-projection modifies net input)")

W_ECout shape : torch.Size([15, 15])  (n_ECout=15, n_units=15)
W_ECout range : [0.4901, 0.5100]
  (expect ~uniform(0.49, 0.51) — Schapiro 2017 SI Table 2 big_loop init)
NET_SCALE ecout_ecin = 2.0  (SI Table 2 abs=2)

Active (no back-proj) : [0, 1]
Active (with ECout[3]): [0, 1]
Outputs differ: True  (expect True — back-projection modifies net input)


In [6]:
# GOAL: Confirm CHL updates W_ECout (big loop) via outer-product difference
# CHL update for W_ECout (big loop)
# ΔW = lr * (outer(a_ECout_plus, a_ECin_plus) − outer(a_ECout_minus, a_ECin_minus))
# O'Reilly & Munakata (2000) Ch. 4 Eq. 4.3; Schapiro (2017) §2.b
a_ECin_m  = clamp_window.clone()   # ECin same both phases (Schapiro 2017 §2.c)
a_ECin_p  = clamp_window.clone()
a_ECout_m = torch.zeros(N_ITEMS); a_ECout_m[2] = 0.8   # free minus-phase prediction
a_ECout_p = torch.zeros(N_ITEMS); a_ECout_p[5] = 1.0   # plus-phase target

W_before = ecin_big.W_ECout.data.clone()
ecin_big.update_weights(
    a_ECout_minus=a_ECout_m, a_ECout_plus=a_ECout_p,
    a_ECin_minus=a_ECin_m,   a_ECin_plus=a_ECin_p,
    lr=0.05,
)
dW = ecin_big.W_ECout.data - W_before

print(f"max |ΔW_ECout| = {dW.abs().max().item():.6f}  (lr=0.05)")
print(f"Non-zero ΔW entries : {(dW != 0).sum().item()}  (expect > 0)")
print("W_ECout updated via CHL: outer(ECout_plus, ECin_plus) − outer(ECout_minus, ECin_minus)")

max |ΔW_ECout| = 0.050000  (lr=0.05)
Non-zero ΔW entries : 4  (expect > 0)
W_ECout updated via CHL: outer(ECout_plus, ECin_plus) − outer(ECout_minus, ECin_minus)


## L_ECout — Forward, Clamp, and CHL

In [7]:
# GOAL: Confirm L_ECout weight shape, k=2 absolute, and Euler state buffers
# Instantiate L_ECout
ecout = L_ECout(n_CA1=N_CA1, n_units=N_ITEMS, k=2, use_euler=True)

print(f"W shape  : {ecout.W.shape}  (n_CA1={N_CA1}, n_units={N_ITEMS})")
print(f"k        : {ecout.k}  (absolute k=2; matches ECin; Schapiro 2017 §2.a.ii)")
print(f"lr       : {ecout.lr}  (MSP rate; Go reimplementation)")
print(f"_Vm shape: {ecout._Vm.shape}  (membrane potential; Euler state)")
print(f"_y shape : {ecout._y.shape}   (firing rate; kWTA output)")

W shape  : torch.Size([50, 15])  (n_CA1=50, n_units=15)
k        : 2  (absolute k=2; matches ECin; Schapiro 2017 §2.a.ii)
lr       : 0.05  (MSP rate; Go reimplementation)
_Vm shape: torch.Size([15])  (membrane potential; Euler state)
_y shape : torch.Size([15])   (firing rate; kWTA output)


In [8]:
# GOAL: Confirm ECout forward pass produces exactly 2 active units from CA1 input
# Forward pass: CA1 activity → ECout settles → k=2 active
# Minus phase: ECout settles freely; activity = CA1's prediction
torch.manual_seed(7)
ecout.W.data = torch.randn(N_CA1, N_ITEMS) * 0.1

# Simulate CA1 output (random, ~10% active)
act_ca1 = torch.zeros(N_CA1)
act_ca1[:5] = torch.rand(5)   # 5 units active (10%)

ecout.reset()
act_ecout = ecout(act_ca1)

n_active = (act_ecout > 0).sum().item()
active_idx = (act_ecout > 0).nonzero().squeeze().tolist()
print(f"Active ECout units: {n_active}  (expect 2; k=2 absolute)")
print(f"Active indices    : {active_idx}")
print(f"Activity values   : {act_ecout[act_ecout > 0].tolist()}")

Active ECout units: 0  (expect 2; k=2 absolute)
Active indices    : []
Activity values   : []


In [9]:
# GOAL: Confirm clamp() sets both _y and _Vm to target (plus-phase clamping for CHL)
# clamp() vs forward(): plus phase sets _y and _Vm to target directly
# Schapiro (2017) §2.b: "in the plus phase, the model is directly shown the correct output"
target = torch.zeros(N_ITEMS)
target[7] = 1.0   # next item = item 7 (one-hot target)

ecout.reset()
act_free = ecout(act_ca1).clone()         # Q2-Q3: free settling
ecout.clamp(target)                        # Q4: clamp to target
act_clamped = ecout._y.clone()

print(f"Free settling (ActM):  {(act_free > 0).nonzero().squeeze().tolist()}")
print(f"After clamp   (ActP):  {(act_clamped > 0).nonzero().squeeze().tolist()}  (expect [7])")
print(f"_y  matches target: {torch.allclose(ecout._y,  target)}")
print(f"_Vm matches target: {torch.allclose(ecout._Vm, target)}")
print()
print("clamp() sets both _y and _Vm. Subsequent Euler steps continue from clamped state.")
print("This is the CHL ActP — the error signal that drives weight updates.")

Free settling (ActM):  []
After clamp   (ActP):  7  (expect [7])
_y  matches target: True
_Vm matches target: True

clamp() sets both _y and _Vm. Subsequent Euler steps continue from clamped state.
This is the CHL ActP — the error signal that drives weight updates.


In [10]:
# GOAL: Confirm CHL updates W_CA1_ECout via outer-product difference across phases
# CHL weight update for W_CA1_ECout
# ΔW = lr * (outer(a_CA1_plus, a_ECout_plus) − outer(a_CA1_minus, a_ECout_minus))
# O'Reilly & Munakata (2000) Ch. 4 Eq. 4.3; Schapiro (2017) §2.b
a_CA1_m   = act_ca1.clone()           # CA1 end of Q2-Q3
a_CA1_p   = act_ca1.clone()           # CA1 end of Q4 (simplified: same CA1 for test)
a_ECout_m = act_free.clone()          # ECout free prediction
a_ECout_p = target.clone()            # ECout clamped to target

W_before = ecout.W.data.clone()
ecout.update_weights(
    a_CA1_minus=a_CA1_m,   a_CA1_plus=a_CA1_p,
    a_ECout_minus=a_ECout_m, a_ECout_plus=a_ECout_p,
)
dW = ecout.W.data - W_before

print(f"max |ΔW_CA1_ECout| = {dW.abs().max().item():.6f}  (lr={ecout.lr})")
print(f"Non-zero ΔW entries: {(dW != 0).sum().item()} / {N_CA1 * N_ITEMS}")
print()
print("Only rows with active CA1 units and cols with active ECout units change.")
print(f"Active CA1 (pre): {(a_CA1_m > 0).sum().item()} units → affects {(a_CA1_m > 0).sum().item()} rows")
print(f"ECout plus (target): item 7 active → strengthens W col 7")
print(f"ECout minus (free) : {(a_ECout_m > 0).nonzero().squeeze().tolist()} active → weakens those cols")

max |ΔW_CA1_ECout| = 0.033490  (lr=0.05)
Non-zero ΔW entries: 5 / 750

Only rows with active CA1 units and cols with active ECout units change.
Active CA1 (pre): 5 units → affects 5 rows
ECout plus (target): item 7 active → strengthens W col 7
ECout minus (free) : [] active → weakens those cols
